# 🫁 LUAD Drug Resistance — Multi-Dataset Integration Pipeline

**Proje:** pARACNe Kinaz Ağı × PRISM Drug Screen × CPTAC Proteomik Entegrasyonu  
**Amaç:** LUAD'da tirozin kinaz aktivasyon örüntülerinin ilaç direnci ile ilişkisini araştırmak  
**Veriler:** PRISM 19Q4 Secondary Screen | CPTAC-LUAD (LinkedOmics) | pARACNe (Bansal et al. 2019)

---
## 📋 İçindekiler
1. Kurulum & Kütüphaneler
2. Veri İndirme
3. PRISM — LUAD Hücre Hattı Filtresi
4. pARACNe Kinaz Ağı Analizi
5. Kinaz–İlaç Korelasyon Analizi (PRISM × CCLE)
6. CPTAC Proteomik Yükleme & Kinaz Skoru
7. Survival Analizi (TCGA-LUAD)
8. Görselleştirme & Sonuçlar


---
## 1. 📦 Kurulum & Kütüphaneler

In [ ]:
# Gerekli kütüphaneleri yükle
!pip install lifelines pydeseq2 gseapy scipy statsmodels openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# Plot ayarları
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
COLORS = {
    'PTK2': '#E63946', 'MET': '#457B9D', 'SRC': '#2A9D8F',
    'EGFR': '#E9C46A', 'LCK': '#F4A261', 'YES1': '#264653'
}

print('✅ Kütüphaneler yüklendi')

---
## 2. 📥 Veri İndirme

> **Manuel yükleme seçeneği:** Dosyaları indirip Google Drive'a koyup mount edebilirsiniz (aşağıda her iki yol da var)

In [ ]:
import os

# ── SEÇENEK A: Google Drive mount (önerilir, büyük dosyalar için) ──
USE_DRIVE = False  # True yaparsanız Drive'dan okur

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/LUAD_Project/'
    print(f'Drive bağlandı: {DATA_DIR}')
else:
    DATA_DIR = '/content/data/'
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f'Yerel klasör: {DATA_DIR}')

In [ ]:
# ── PRISM 19Q4 Secondary Screen İndir ──
# Kaynak: https://depmap.org/portal/data_page/?tab=allData

PRISM_FILES = {
    'cell_info': 'https://ndownloader.figshare.com/files/20237715',
    'drug_meta': 'https://ndownloader.figshare.com/files/20237769',
    'auc_matrix': 'https://ndownloader.figshare.com/files/20237757',  # secondary-screen-dose-response-curve-parameters
}

# Not: DepMap figshare URL'leri değişebilir. Çalışmıyorsa manuel yükleme yapın.
# Manuel yükleme: depmap.org/portal/data_page/ → PRISM Repurposing 19Q4

print('⚠️  PRISM dosyaları büyüktür (~150-500 MB)')
print('Önerim: depmap.org/portal/data_page/ adresinden indirip Drive\'a yükleyin')
print()
print('İndirilmesi gereken dosyalar:')
print('  1. secondary-screen-dose-response-curve-parameters.csv  (~150 MB)')
print('  2. secondary-screen-cell-line-info.csv                  (<1 MB)')
print('  3. repurposing_samples.csv                              (<5 MB)')

In [ ]:
# ── CPTAC-LUAD LinkedOmics'ten İndir (otomatik) ──

CPTAC_URLS = {
    'proteome_tumor': 'https://linkedomics.org/data_download/CPTAC-LUAD/HS_CPTAC_LUAD_proteome_ratio_NArm_TUMOR.cct',
    'phospho_tumor':  'https://linkedomics.org/data_download/CPTAC-LUAD/HS_CPTAC_LUAD_phosphoproteome_ratio_norm_NArm_TUMOR.cct',
    'clinical':       'https://linkedomics.org/data_download/CPTAC-LUAD/HS_CPTAC_LUAD_cli.tsi',
    'mutation':       'https://linkedomics.org/data_download/CPTAC-LUAD/HS_CPTAC_LUAD_somatic_mutation_gene.cbt',
    'cnv':            'https://linkedomics.org/data_download/CPTAC-LUAD/HS_CPTAC_LUAD_cnv_gene_LR.cct',
}

for name, url in CPTAC_URLS.items():
    fpath = os.path.join(DATA_DIR, f'cptac_{name}.txt')
    if not os.path.exists(fpath):
        print(f'  İndiriliyor: {name}...')
        !wget -q -O "{fpath}" "{url}"
        print(f'  ✅ {name} ({os.path.getsize(fpath)/1024/1024:.1f} MB)')
    else:
        print(f'  ✔ {name} zaten mevcut')

print('\n✅ CPTAC verileri hazır')

In [ ]:
# ── pARACNe Verisini Yükle ──
# Dosyaları ZIP'ten çıkarıp DATA_DIR'e koyun: S1_peptide_interactions.csv, S2_protein_interactions.csv

from google.colab import files

print('pARACNe dosyalarını yükleyin (S1 ve S2 CSV\'leri):')
print('Zip dosyanızı açıp CSV dosyalarını yükleyebilirsiniz.')

# Yükleme:
# uploaded = files.upload()
# for fn, content in uploaded.items():
#     with open(os.path.join(DATA_DIR, fn), 'wb') as f:
#         f.write(content)

---
## 3. 🔬 PRISM — LUAD Hücre Hattı Filtresi

In [ ]:
# Hücre hattı bilgisini yükle
cell_info_path = os.path.join(DATA_DIR, 'secondary-screen-cell-line-info.csv')

cell_info = pd.read_csv(cell_info_path)
print(f'Toplam hücre hattı: {len(cell_info)}')
print(f'Kolonlar: {list(cell_info.columns[:10])}')
cell_info.head()

In [ ]:
# LUAD hücre hatlarını filtrele
# Kolon adı farklı olabilir: 'lineage', 'disease', 'primary_disease', 'lineage_subtype'

# Hangi kolon kanser tipini içeriyor?
for col in cell_info.columns:
    sample = cell_info[col].astype(str).str.lower()
    if sample.str.contains('lung|luad|adenocarcinoma', na=False).any():
        print(f'✅ Kanser tipi kolonu bulundu: [{col}]')
        print(cell_info[col].value_counts().head(10))
        LINEAGE_COL = col
        break

In [ ]:
# LUAD hücre hatlarını seç
luad_mask = (
    cell_info[LINEAGE_COL].astype(str).str.lower().str.contains('lung', na=False)
)

# Akciğer adenokarsinomunu spesifik olarak filtrele (varsa)
if 'lineage_subtype' in cell_info.columns:
    luad_mask = luad_mask & (
        cell_info['lineage_subtype'].astype(str).str.lower().str.contains('adenocarcinoma|luad', na=False)
    )

luad_cells = cell_info[luad_mask].copy()
print(f'LUAD hücre hattı sayısı: {len(luad_cells)}')

# DepMap ID'lerini al
ID_COL = 'depmap_id' if 'depmap_id' in cell_info.columns else cell_info.columns[0]
luad_ids = luad_cells[ID_COL].tolist()
print(f'LUAD IDs örnek: {luad_ids[:5]}')

In [ ]:
# AUC matrisini yükle (büyük dosya — ilk satırları göster)
auc_path = os.path.join(DATA_DIR, 'secondary-screen-dose-response-curve-parameters.csv')

print('AUC matrisi yükleniyor (bu biraz sürebilir)...')
prism_auc = pd.read_csv(auc_path, index_col=0, low_memory=False)
print(f'AUC matrisi boyutu: {prism_auc.shape}  (hücre hatları × ilaçlar)')
prism_auc.iloc[:5, :5]

In [ ]:
# AUC matrisini LUAD ile filtrele
# Matris formatına göre index veya kolon olabilir
if prism_auc.index[0] in luad_ids:
    luad_auc = prism_auc.loc[prism_auc.index.isin(luad_ids)]
elif prism_auc.columns[0] in luad_ids:
    luad_auc = prism_auc[prism_auc.columns.intersection(luad_ids)].T
else:
    # Alternatif: satır isimlerinde ara
    luad_auc = prism_auc.loc[prism_auc.index.isin(luad_ids)]

print(f'LUAD AUC matrisi: {luad_auc.shape}  (hücre × ilaç)')
luad_auc.head()

---
## 4. 🕸️ pARACNe Kinaz Ağı Analizi

In [ ]:
# S2 protein-level etkileşimlerini yükle
s2_path = os.path.join(DATA_DIR, 'S2_protein_interactions.csv')

try:
    s2 = pd.read_csv(s2_path)
    print(f'S2 yüklendi: {s2.shape}')
    print(f'Kolonlar: {list(s2.columns)}')
    s2.head()
except FileNotFoundError:
    print('⚠️  S2 dosyası bulunamadı. Lütfen pARACNe ZIP içindeki CSV\'yi yükleyin.')
    print('Yukarıdaki hücrede files.upload() kodunu çalıştırın.')

In [ ]:
# Kolon isimlerini normalize et
# pARACNe S2'de beklenen kolonlar: kinase, substrate, MI/weight
col_map = {}
for c in s2.columns:
    cl = c.lower()
    if 'kinas' in cl or 'regulator' in cl or 'source' in cl:
        col_map[c] = 'kinase'
    elif 'substrate' in cl or 'target' in cl or 'gene' in cl:
        col_map[c] = 'substrate'
    elif 'mi' in cl or 'weight' in cl or 'score' in cl:
        col_map[c] = 'MI'

s2 = s2.rename(columns=col_map)
print('Kolon eşleştirmesi:', col_map)
s2.head()

In [ ]:
# Kinaz hub analizi — her kinazın substrat sayısı
hub_kinases = s2.groupby('kinase')['substrate'].count().sort_values(ascending=False)

print(f'Toplam benzersiz kinaz: {len(hub_kinases)}')
print(f'\nTop 15 Hub Kinaz:')
print(hub_kinases.head(15).to_string())

# Odak kinazlar (projeniz için kritik)
FOCUS_KINASES = ['PTK2', 'MET', 'EGFR', 'SRC', 'LCK', 'YES1', 'FYN', 'LYN', 'DDR1']
focus_present = [k for k in FOCUS_KINASES if k in hub_kinases.index]
print(f'\nOdak kinazlar ağda mevcut: {focus_present}')

In [ ]:
# Hub kinaz görselleştirmesi
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sol: Tüm kinazlar barplot
top20 = hub_kinases.head(20)
colors_bar = ['#E63946' if k in FOCUS_KINASES else '#ADB5BD' for k in top20.index]
axes[0].barh(top20.index[::-1], top20.values[::-1], color=colors_bar[::-1])
axes[0].set_xlabel('Substrat Sayısı', fontsize=12)
axes[0].set_title('pARACNe — Top 20 Hub Kinaz\n(kırmızı = odak kinazlar)', fontsize=13)
axes[0].axvline(100, ls='--', color='gray', alpha=0.5, label='n=100')

# Sağ: Sadece odak kinazlar
focus_data = hub_kinases.reindex(focus_present).dropna().sort_values(ascending=False)
focus_colors = [COLORS.get(k, '#6C757D') for k in focus_data.index]
axes[1].bar(focus_data.index, focus_data.values, color=focus_colors, edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('Substrat Sayısı', fontsize=12)
axes[1].set_title('Odak Kinazlar — pARACNe Network', fontsize=13)
axes[1].tick_params(axis='x', rotation=30)

for i, (k, v) in enumerate(zip(focus_data.index, focus_data.values)):
    axes[1].text(i, v + 2, str(v), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'fig1_hub_kinases.png'), dpi=200, bbox_inches='tight')
plt.show()
print('✅ Şekil kaydedildi: fig1_hub_kinases.png')

In [ ]:
# Her odak kinazın substratlarını al
kinase_substrates = {}
for k in focus_present:
    subs = s2[s2['kinase'] == k]['substrate'].tolist()
    kinase_substrates[k] = subs
    print(f'{k}: {len(subs)} substrat  |  örnek: {subs[:5]}')

---
## 5. 💊 Kinaz–İlaç Korelasyon Analizi (PRISM × CCLE Expression)

In [ ]:
# İlaç metadata'sını yükle — hangi ilaç hangi kinazı hedef alıyor?
drug_meta_path = os.path.join(DATA_DIR, 'repurposing_samples.csv')

drug_meta = pd.read_csv(drug_meta_path)
print(f'İlaç metadata: {drug_meta.shape}')
print(f'Kolonlar: {list(drug_meta.columns)}')
drug_meta.head()

In [ ]:
# Kinaz inhibitörlerini filtrele
# Hedef kolonunu bul
target_col = None
for col in drug_meta.columns:
    if 'target' in col.lower() or 'moa' in col.lower():
        target_col = col
        break

if target_col:
    print(f'Hedef kolonu: [{target_col}]')
    
    # PTK2/FAK, MET, EGFR, SRC inhibitörleri
    ki_mask = drug_meta[target_col].astype(str).str.upper().str.contains(
        'PTK2|FAK|MET|EGFR|SRC|ALK|MET|LCK|YES|FYN|DDR', na=False
    )
    kinase_inhibitors = drug_meta[ki_mask].copy()
    print(f'Kinaz inhibitörü sayısı: {len(kinase_inhibitors)}')
    print(kinase_inhibitors[[target_col, 'name' if 'name' in drug_meta.columns else drug_meta.columns[1]]].head(20))
else:
    print('⚠️  Target kolonu bulunamadı. Kolonları kontrol edin.')

In [ ]:
# CCLE mRNA expression verisi — LUAD hücre hatları için kinaz expression
# Kaynak: depmap.org → CCLE_expression.csv (büyük dosya, Drive'dan yükleyin)
# Alternatif: cBioPortal CCLE verisini kullanabilirsiniz

ccle_path = os.path.join(DATA_DIR, 'CCLE_expression.csv')

if os.path.exists(ccle_path):
    print('CCLE expression yükleniyor...')
    ccle_expr = pd.read_csv(ccle_path, index_col=0)
    
    # LUAD hücre hatlarını filtrele
    ccle_luad = ccle_expr.loc[ccle_expr.index.isin(luad_ids)]
    print(f'CCLE LUAD: {ccle_luad.shape}')
    
    # Kinaz kolonlarını seç
    kinase_cols = [c for c in ccle_luad.columns if any(k in c for k in FOCUS_KINASES)]
    ccle_kinase = ccle_luad[kinase_cols]
    print(f'Kinaz expression kolonları: {kinase_cols}')
else:
    print('ℹ️  CCLE_expression.csv bulunamadı.')
    print('İndirme linki: https://depmap.org/portal/data_page/?tab=allData')
    print('→ Arama: "CCLE_expression" → DepMap 24Q2')

In [ ]:
# Kinaz expression × İlaç AUC Korelasyon Analizi
# CCLE ve PRISM mevcut olduğunda çalışır

def kinase_drug_correlation(kinase_name, ccle_kinase_expr, drug_auc_series, min_samples=10):
    """
    Bir kinazın expression düzeyi ile ilaç AUC değeri arasındaki Spearman korelasyonunu hesaplar.
    
    Parameters:
    -----------
    kinase_name : str — kinaz gen adı
    ccle_kinase_expr : pd.Series — kinaz expression (index=depmap_id)
    drug_auc_series  : pd.Series — ilaç AUC değerleri (index=depmap_id)
    
    Returns:
    --------
    dict: rho, pval, n, kinase, drug
    """
    common = ccle_kinase_expr.index.intersection(drug_auc_series.index)
    if len(common) < min_samples:
        return None
    
    x = ccle_kinase_expr.loc[common].dropna()
    y = drug_auc_series.loc[x.index].dropna()
    common2 = x.index.intersection(y.index)
    
    if len(common2) < min_samples:
        return None
    
    rho, pval = spearmanr(x.loc[common2], y.loc[common2])
    return {'kinase': kinase_name, 'rho': rho, 'pval': pval, 'n': len(common2)}


def batch_kinase_drug_correlation(ccle_df, prism_df, kinase_list, drug_id_list):
    """
    Tüm kinaz × ilaç kombinasyonları için korelasyon hesapla.
    """
    results = []
    for kinase in kinase_list:
        if kinase not in ccle_df.columns:
            continue
        kinase_expr = ccle_df[kinase]
        
        for drug_id in drug_id_list:
            if drug_id not in prism_df.columns:
                continue
            drug_auc = prism_df[drug_id]
            
            res = kinase_drug_correlation(kinase, kinase_expr, drug_auc)
            if res:
                res['drug_id'] = drug_id
                results.append(res)
    
    if not results:
        return pd.DataFrame()
    
    df = pd.DataFrame(results)
    # FDR düzeltmesi
    _, df['fdr'], _, _ = multipletests(df['pval'], method='fdr_bh')
    return df.sort_values('fdr')


print('✅ Korelasyon fonksiyonları tanımlandı')
print('Çalıştırmak için: batch_kinase_drug_correlation(ccle_kinase, luad_auc, FOCUS_KINASES, ki_drug_ids)')

In [ ]:
# Korelasyon heatmap çizim fonksiyonu
def plot_kinase_drug_heatmap(corr_df, top_n=15, title='Kinaz × İlaç Spearman Korelasyonu'):
    """
    Kinaz-ilaç korelasyon matrisini heatmap olarak görselleştirir.
    """
    if corr_df.empty:
        print('Veri yok — önce korelasyon analizi çalıştırın')
        return
    
    # Pivot tablo oluştur
    pivot = corr_df.pivot_table(index='kinase', columns='drug_id', values='rho')
    
    # En anlamlı ilişkilere sahip ilaçları seç
    top_drugs = corr_df[corr_df['fdr'] < 0.05].nlargest(top_n, 'rho')['drug_id'].unique()
    if len(top_drugs) == 0:
        top_drugs = corr_df.nlargest(top_n, 'rho')['drug_id'].unique()
    
    pivot_sub = pivot[pivot.columns.intersection(top_drugs)]
    
    fig, ax = plt.subplots(figsize=(max(10, len(top_drugs)*0.8), max(5, len(pivot_sub)*0.6)))
    sns.heatmap(
        pivot_sub,
        cmap='RdBu_r',
        center=0,
        vmin=-0.6, vmax=0.6,
        annot=True, fmt='.2f',
        linewidths=0.5,
        ax=ax
    )
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_xlabel('İlaç ID', fontsize=11)
    ax.set_ylabel('Kinaz', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, 'fig2_kinase_drug_heatmap.png'), dpi=200, bbox_inches='tight')
    plt.show()

print('✅ Heatmap fonksiyonu tanımlandı')

---
## 6. 🧬 CPTAC Proteomik — Kinaz Aktivasyon Skoru

In [ ]:
# CPTAC proteom yükle
proteome = pd.read_csv(os.path.join(DATA_DIR, 'cptac_proteome_tumor.txt'), sep='\t', index_col=0)
print(f'Proteom boyutu: {proteome.shape}  (genler × hastalar)')
proteome.iloc[:5, :5]

In [ ]:
# Fosforproteom yükle
phospho = pd.read_csv(os.path.join(DATA_DIR, 'cptac_phospho_tumor.txt'), sep='\t', index_col=0)
print(f'Fosforproteom boyutu: {phospho.shape}  (fosfosite × hastalar)')

# Klinik veri
clinical = pd.read_csv(os.path.join(DATA_DIR, 'cptac_clinical.txt'), sep='\t', index_col=0)
print(f'Klinik veri: {clinical.shape}')
print(f'Klinik kolonlar: {list(clinical.columns)}')

In [ ]:
# Kinaz Aktivasyon Skoru hesapla
# Her hasta için: kinazın substratlarının ortalama fosforilasyon düzeyi

def compute_kinase_activation_score(phospho_df, substrates, min_substrates=5):
    """
    Fosforproteomik veriden kinaz aktivasyon skoru hesaplar.
    
    Yöntem: Kinazın pARACNe substratlarının ortalama fosforilasyon düzeyi.
    
    Parameters:
    -----------
    phospho_df : pd.DataFrame — fosfosite × hasta matrisi
    substrates : list — kinazın substrat gen listesi
    
    Returns:
    --------
    pd.Series — hasta başına kinaz aktivasyon skoru
    """
    # Fosfosite isimlerinden gen adını çıkar (örn: 'EGFR_Y1068' → 'EGFR')
    phospho_genes = phospho_df.index.str.split('_').str[0]
    
    # Substratlarla eşleşen fosforsite satırlarını bul
    mask = phospho_genes.isin(substrates)
    sub_phospho = phospho_df.loc[mask]
    
    if len(sub_phospho) < min_substrates:
        return None
    
    # Hasta başına ortalama fosforilasyon
    score = sub_phospho.mean(axis=0)
    return score


# Her odak kinaz için skor hesapla
kinase_scores = {}
for kinase in focus_present:
    substrates = kinase_substrates.get(kinase, [])
    score = compute_kinase_activation_score(phospho, substrates)
    if score is not None:
        kinase_scores[kinase] = score
        print(f'✅ {kinase}: {len(substrates)} substrat → skor hesaplandı ({score.notna().sum()} hasta)')
    else:
        print(f'⚠️  {kinase}: yetersiz substrat eşleşmesi')

kinase_score_df = pd.DataFrame(kinase_scores)
print(f'\nKinaz skor matrisi: {kinase_score_df.shape}  (hastalar × kinazlar)')

In [ ]:
# Kinaz skor dağılımı — violin plot
fig, ax = plt.subplots(figsize=(12, 5))

kinase_score_melted = kinase_score_df.melt(var_name='Kinaz', value_name='Aktivasyon Skoru')
palette = {k: COLORS.get(k, '#ADB5BD') for k in kinase_score_df.columns}

sns.violinplot(
    data=kinase_score_melted,
    x='Kinaz', y='Aktivasyon Skoru',
    palette=palette,
    inner='box',
    ax=ax
)
ax.axhline(0, ls='--', color='black', alpha=0.4)
ax.set_title('CPTAC-LUAD: Kinaz Aktivasyon Skoru Dağılımı (n=110)', fontsize=13)
ax.set_ylabel('Ortalama Substrat Fosforilasyon (Log2ratio)', fontsize=11)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'fig3_kinase_activation_scores.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# pARACNe kinaz-substrat ilişkilerini CPTAC fosforproteomik verisiyle doğrula
# Her kinaz-substrat çifti: kinaz protein abundance × substrat fosforilasyon korelasyonu

def validate_kinase_substrate(kinase, substrate, proteome_df, phospho_df):
    """
    Kinaz proteomik ekspresyon ile substrat fosforilasyonu arasındaki korelasyonu hesaplar.
    Yüksek korelasyon → pARACNe tahmini doğrulandı
    """
    # Kinaz protein abundance
    if kinase not in proteome_df.index:
        return None
    kinase_prot = proteome_df.loc[kinase]
    
    # Substrat fosfosite'leri bul
    phospho_genes = phospho_df.index.str.split('_').str[0]
    sub_mask = phospho_genes == substrate
    if sub_mask.sum() == 0:
        return None
    
    results = []
    for site in phospho_df.index[sub_mask]:
        sub_phospho = phospho_df.loc[site]
        common = kinase_prot.index.intersection(sub_phospho.index)
        if len(common) < 10:
            continue
        rho, pval = spearmanr(kinase_prot[common].fillna(0), sub_phospho[common].fillna(0))
        results.append({'kinase': kinase, 'substrate': substrate, 'site': site, 
                        'rho': rho, 'pval': pval, 'n': len(common)})
    
    return pd.DataFrame(results) if results else None


# Odak kinazlar için top substratları doğrula
validation_results = []
for kinase in focus_present:
    subs = kinase_substrates[kinase][:20]  # En fazla 20 substrat
    for sub in subs:
        res = validate_kinase_substrate(kinase, sub, proteome, phospho)
        if res is not None and len(res) > 0:
            validation_results.append(res)

if validation_results:
    val_df = pd.concat(validation_results, ignore_index=True)
    _, val_df['fdr'], _, _ = multipletests(val_df['pval'], method='fdr_bh')
    val_df = val_df.sort_values('fdr')
    
    sig = val_df[val_df['fdr'] < 0.05]
    print(f'Doğrulanan kinaz-substrat çifti (FDR<0.05): {len(sig)} / {len(val_df)}')
    print(f'Doğrulama oranı: {len(sig)/len(val_df)*100:.1f}%')
    val_df.head(20)
else:
    print('ℹ️  Doğrulama için önce proteom ve fosforproteom verilerini yükleyin.')

---
## 7. 📊 Survival Analizi (TCGA-LUAD)

> **TCGA verisi:** cBioPortal'dan indirin → https://www.cbioportal.org/study/summary?id=luad_tcga_pan_can_atlas_2018

In [ ]:
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# TCGA klinik veri yükle (cBioPortal'dan indirilen)
tcga_clinical_path = os.path.join(DATA_DIR, 'tcga_luad_clinical.txt')
tcga_expr_path     = os.path.join(DATA_DIR, 'tcga_luad_mrna.txt')

if os.path.exists(tcga_clinical_path) and os.path.exists(tcga_expr_path):
    tcga_clin = pd.read_csv(tcga_clinical_path, sep='\t', comment='#')
    tcga_expr = pd.read_csv(tcga_expr_path, sep='\t', index_col=0)
    print(f'TCGA klinik: {tcga_clin.shape}')
    print(f'TCGA expression: {tcga_expr.shape}')
    print(f'Klinik kolonlar: {list(tcga_clin.columns[:15])}')
else:
    print('⚠️  TCGA dosyaları bulunamadı.')
    print('İndirme: cbioportal.org → TCGA LUAD PanCancer Atlas → Data → Clinical + mRNA')

In [ ]:
def kaplan_meier_by_kinase(kinase_name, tcga_expr_df, tcga_clin_df,
                            os_col='OS_MONTHS', status_col='OS_STATUS',
                            cutoff='median'):
    """
    Kinaz expression düzeyine göre (yüksek/düşük) KM eğrisi çizer.
    
    Parameters:
    -----------
    kinase_name : str
    cutoff : 'median' veya float (0-1 arası yüzdelik)
    """
    if kinase_name not in tcga_expr_df.index:
        print(f'⚠️  {kinase_name} expression matrisinde bulunamadı')
        return
    
    expr = tcga_expr_df.loc[kinase_name]
    
    # Klinik ile birleştir
    merged = tcga_clin_df.copy()
    merged[kinase_name] = merged.index.map(expr)
    merged = merged.dropna(subset=[os_col, status_col, kinase_name])
    
    # Survival event encode
    if merged[status_col].dtype == object:
        merged['event'] = merged[status_col].str.contains('DECEASED|1|Dead', case=False).astype(int)
    else:
        merged['event'] = merged[status_col]
    
    # Cutoff
    threshold = merged[kinase_name].median() if cutoff == 'median' else merged[kinase_name].quantile(cutoff)
    merged['group'] = np.where(merged[kinase_name] >= threshold, 'High', 'Low')
    
    # Log-rank test
    high = merged[merged['group'] == 'High']
    low  = merged[merged['group'] == 'Low']
    lr = logrank_test(high[os_col], low[os_col], high['event'], low['event'])
    
    # KM plot
    fig, ax = plt.subplots(figsize=(8, 5))
    kmf = KaplanMeierFitter()
    
    for grp, color, ls in [('High', '#E63946', '-'), ('Low', '#457B9D', '--')]:
        d = merged[merged['group'] == grp]
        kmf.fit(d[os_col], d['event'], label=f'{grp} (n={len(d)})')
        kmf.plot_survival_function(ax=ax, color=color, linestyle=ls, linewidth=2, ci_show=True, ci_alpha=0.1)
    
    ax.set_title(f'{kinase_name} Ekspresyon — Overall Survival (TCGA-LUAD)\n'
                 f'Log-rank p = {lr.p_value:.4f}', fontsize=12)
    ax.set_xlabel('Süre (Ay)', fontsize=11)
    ax.set_ylabel('Sağkalım Olasılığı', fontsize=11)
    ax.legend(fontsize=10)
    ax.set_ylim(0, 1.05)
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, f'fig4_KM_{kinase_name}.png'), dpi=200, bbox_inches='tight')
    plt.show()
    
    print(f'{kinase_name}: High={len(high)}, Low={len(low)}, Log-rank p={lr.p_value:.4f}')
    return lr.p_value


print('✅ Kaplan-Meier fonksiyonu tanımlandı')
print('Çalıştırmak için: kaplan_meier_by_kinase("PTK2", tcga_expr, tcga_clin)')

In [ ]:
# Tüm odak kinazlar için KM analizi
if 'tcga_expr' in dir() and 'tcga_clin' in dir():
    km_results = {}
    for kinase in focus_present:
        pval = kaplan_meier_by_kinase(kinase, tcga_expr, tcga_clin)
        km_results[kinase] = pval
    
    # Özet tablo
    km_summary = pd.DataFrame.from_dict(km_results, orient='index', columns=['logrank_p'])
    km_summary['significant'] = km_summary['logrank_p'] < 0.05
    print('\n=== KM Özet ===')
    print(km_summary.sort_values('logrank_p'))
else:
    print('ℹ️  TCGA verisi yüklü değil — önce tcga_clinical ve tcga_mrna dosyalarını yükleyin')

In [ ]:
# Çok Değişkenli Cox Regresyon
def cox_multivariate(kinases, tcga_expr_df, tcga_clin_df,
                     os_col='OS_MONTHS', status_col='OS_STATUS',
                     covariates=['AGE', 'STAGE']):
    """
    Kinaz ekspresyon düzeylerini covariat olarak kullanan çok değişkenli Cox modeli.
    """
    merged = tcga_clin_df.copy()
    
    # Kinaz expression ekle
    for k in kinases:
        if k in tcga_expr_df.index:
            merged[k] = merged.index.map(tcga_expr_df.loc[k])
    
    # Event encode
    if merged[status_col].dtype == object:
        merged['event'] = merged[status_col].str.contains('DECEASED|1|Dead', case=False).astype(int)
    else:
        merged['event'] = merged[status_col]
    
    # Model kolonları
    model_cols = [os_col, 'event'] + kinases + [c for c in covariates if c in merged.columns]
    model_df = merged[model_cols].dropna()
    
    cph = CoxPHFitter()
    cph.fit(model_df, duration_col=os_col, event_col='event')
    cph.print_summary()
    
    # Forest plot
    cph.plot(ax=plt.gca())
    plt.title('Cox Multivariate — HR (95% CI)', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, 'fig5_cox_forest_plot.png'), dpi=200, bbox_inches='tight')
    plt.show()
    
    return cph


print('✅ Cox regresyon fonksiyonu tanımlandı')
# Çalıştırmak için:
# cph = cox_multivariate(focus_present, tcga_expr, tcga_clin)

---
## 8. 📈 Sonuç Özeti & Dışa Aktarma

In [ ]:
# Tüm sonuçları Excel'e aktar

output_excel = os.path.join(DATA_DIR, 'LUAD_Analysis_Results.xlsx')

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    
    # 1. Hub kinaz analizi
    hub_kinases.reset_index().rename(columns={'kinase': 'Kinaz', 'substrate': 'Substrat Sayısı'}) \
        .to_excel(writer, sheet_name='Hub_Kinazlar', index=False)
    
    # 2. Kinaz aktivasyon skorları
    if 'kinase_score_df' in dir() and not kinase_score_df.empty:
        kinase_score_df.to_excel(writer, sheet_name='KAS_CPTAC')
    
    # 3. Korelasyon sonuçları
    if 'val_df' in dir() and not val_df.empty:
        val_df.to_excel(writer, sheet_name='pARACNe_Dogrulama', index=False)
    
    # 4. KM sonuçları
    if 'km_summary' in dir():
        km_summary.to_excel(writer, sheet_name='KM_Survival')

print(f'✅ Sonuçlar kaydedildi: {output_excel}')

In [ ]:
# Özet panel — tüm figürleri tek ekranda
print('=' * 60)
print('         LUAD DRUG RESISTANCE ANALYSIS — ÖZET')
print('=' * 60)
print()
print(f'📊 pARACNe Ağı:')
print(f'   Toplam kinaz : {len(hub_kinases)}')
print(f'   Odak kinaz   : {len(focus_present)}')
print(f'   Top kinaz    : {hub_kinases.index[0]} ({hub_kinases.iloc[0]} substrat)')
print()

if 'kinase_score_df' in dir() and not kinase_score_df.empty:
    print(f'🧬 CPTAC Kinaz Aktivasyon Skoru:')
    print(f'   Hasta sayısı : {len(kinase_score_df)}')
    for k in kinase_score_df.columns:
        m = kinase_score_df[k].median()
        print(f'   {k:10s} median: {m:.3f}')
    print()

if 'val_df' in dir() and not val_df.empty:
    sig_rate = len(val_df[val_df['fdr'] < 0.05]) / len(val_df) * 100
    print(f'✅ pARACNe Doğrulama Oranı (CPTAC): {sig_rate:.1f}%')
    print()

if 'km_summary' in dir():
    print(f'📉 Survival Analizi:')
    sig_km = km_summary[km_summary['significant']]
    print(f'   Anlamlı kinaz (p<0.05): {len(sig_km)} / {len(km_summary)}')
    for k in sig_km.index:
        print(f'   {k}: p={sig_km.loc[k, "logrank_p"]:.4f}')

print()
print('Kaydedilen figürler:')
for f in os.listdir(DATA_DIR):
    if f.endswith('.png'):
        print(f'  📷 {f}')

In [ ]:
# Tüm çıktı dosyalarını zip'le ve indir
import shutil

zip_name = '/content/LUAD_Analysis_Output'
shutil.make_archive(zip_name, 'zip', DATA_DIR)
print(f'✅ ZIP hazır: {zip_name}.zip')

# İndirmek için:
from google.colab import files
files.download(f'{zip_name}.zip')

---
## 📚 Referanslar

1. **pARACNe:** Bansal M, et al. *A community computational challenge to predict the activity of pairs of compounds.* PLoS One. 2019. PMID: 30615629
2. **PRISM:** Corsello SM, et al. *Discovering the anti-cancer potential of non-oncology drugs by systematic viability profiling.* Nature Cancer. 2020.
3. **CPTAC-LUAD:** Gillette MA, et al. *Proteogenomic Characterization Reveals Therapeutic Vulnerabilities in Lung Adenocarcinoma.* Cell. 2020.
4. **LinkedOmics:** Vasaikar SV, et al. *LinkedOmics: analyzing multi-omics data within and across 32 cancer types.* Nucleic Acids Res. 2018.

---
*Notebook: LUAD Drug Resistance Multi-Dataset Integration Pipeline*  
*Oluşturma tarihi: Mart 2026*